# Aktivasyon Fonksiyonlarının Eğitim Dinamiklerine Etkisi
## Effect of Activation Functions on Training Dynamics

**Ders:** SWE012 – Deep Learning with Python  
**Konu:** Aktivasyon fonksiyonlarının (Sigmoid, Tanh, ReLU, Leaky ReLU) eğitim süreci üzerindeki etkilerinin kontrollü deneylerle incelenmesi.  

**T-Model Yaklaşımı:**
- **Derinlik:** Aktivasyon fonksiyonları üzerine sistematik, kontrollü deneyler
- **Genişlik:** Derste işlenen tüm metodolojilerin (ML temelleri, regularization, optimization, initialization) aktivasyon fonksiyonlarıyla etkileşimi

---

## 1. Kütüphaneler ve Kurulum

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. Veri Seti: Fashion-MNIST

Fashion-MNIST, 10 sınıflı bir görüntü sınıflandırma problemidir (28×28 gri tonlamalı).  
- **i.i.d. varsayımı:** Train ve test setleri aynı dağılımdan bağımsız örneklenmiştir.  
- **Generalization:** Modelin train'de değil, **test** setindeki performansı asıl ölçümüzdür.  
- **Validation set:** Hyperparameter tuning ve early stopping için train setinden ayrılır (data leakage yok).

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

full_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

n = len(full_train)
n_val = int(n * 0.1)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42))
train_dataset = Subset(full_train, indices[:n - n_val])
val_dataset = Subset(full_train, indices[n - n_val:])

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Sinif sayisi: {len(class_names)} | Giris boyutu: 28x28 = 784")
print(f"Minibatch boyutu: {BATCH_SIZE} -> Epoch basina ~{len(train_loader)} iterasyon")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = full_train[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label])
    ax.axis('off')
plt.suptitle('Fashion-MNIST Ornekleri', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model Mimarisi ve Eğitim Altyapısı

**Feedforward Neural Network:** Giriş → Gizli Katmanlar → Softmax Çıkış  
- **Softmax + CrossEntropyLoss:** 10 sınıflı sınıflandırma için standart eşleşme (logit'ler doğrudan verilir, softmax loss içinde)  
- **MLE bağlantısı:** Cross-entropy loss minimize etmek = Categorical dağılım altında MLE yapmak  
- **Backpropagation:** Chain rule + memoization ile gradyanlar hesaplanır  
- **Mimari:** 784 → 256 → 128 → 64 → 10 (3 gizli katman)

In [ ]:
ACTIVATION_FUNCTIONS = {
    'Sigmoid': nn.Sigmoid,
    'Tanh': nn.Tanh,
    'ReLU': nn.ReLU,
    'LeakyReLU': lambda: nn.LeakyReLU(0.01),
}

COLORS = {'Sigmoid': '#e74c3c', 'Tanh': '#3498db', 'ReLU': '#2ecc71', 'LeakyReLU': '#f39c12'}


def build_model(activation_name, hidden_sizes=[256, 128, 64], use_dropout=False,
                use_batchnorm=False, dropout_p=0.3, input_size=784, num_classes=10):
    layers = []
    in_features = input_size
    for h in hidden_sizes:
        layers.append(nn.Linear(in_features, h))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(h))
        act_fn = ACTIVATION_FUNCTIONS[activation_name]
        layers.append(act_fn() if not callable(act_fn) or act_fn in (nn.Sigmoid, nn.Tanh, nn.ReLU) else act_fn())
        if use_dropout:
            layers.append(nn.Dropout(dropout_p))
        in_features = h
    layers.append(nn.Linear(in_features, num_classes))
    return nn.Sequential(*layers)


def init_weights(model, method='he'):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if method == 'he':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif method == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif method == 'random':
                nn.init.normal_(m.weight, 0, 0.5)
            elif method == 'zeros':
                nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)
    return model

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion,
                epochs=15, scheduler=None, early_stop_patience=0, grad_clip=None):
    model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'grad_norms': []}
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        epoch_grad_norms = []
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()

            if grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            total_norm = sum(p.grad.norm(2).item()**2 for p in model.parameters() if p.grad is not None)**0.5
            epoch_grad_norms.append(total_norm)

            optimizer.step()
            running_loss += loss.item()

        if scheduler:
            scheduler.step()

        train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.view(-1, 784).to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                val_loss += criterion(logits, y_batch).item()
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        val_acc = correct / total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['grad_norms'].append(np.mean(epoch_grad_norms))

        if early_stop_patience > 0:
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

    return history


def evaluate_test(model, test_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)
            correct += (model(X_batch).argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

In [ ]:
def plot_results(results_dict, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for name, hist in results_dict.items():
        c = COLORS.get(name.split(' ')[0], None)
        axes[0].plot(hist['train_loss'], label=name, color=c)
        axes[1].plot(hist['val_loss'], label=name, color=c)
        axes[2].plot(hist['val_acc'], label=name, color=c)
    axes[0].set_title(f'{title} — Train Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
    axes[1].set_title(f'{title} — Val Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
    axes[2].set_title(f'{title} — Val Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy'); axes[2].legend()
    plt.tight_layout(); plt.show()


def summary_table(results_dict):
    print(f"{'Config':<35} {'Train Loss':>12} {'Val Loss':>12} {'Best Val Acc':>14}")
    print('-' * 75)
    for name, hist in results_dict.items():
        best_acc = max(hist['val_acc'])
        print(f"{name:<35} {hist['train_loss'][-1]:>12.4f} {hist['val_loss'][-1]:>12.4f} {best_acc:>13.4f}")

---
## 4. DENEY 1: Aktivasyon Fonksiyonu Karşılaştırması (Derinlik)

**Kontrollü Deney Tasarımı:** Tüm parametreler sabit, yalnızca aktivasyon fonksiyonu değişiyor.  
- Mimari: 784 → 256 → 128 → 64 → 10  
- Optimizer: Adam (lr=0.001) — endüstri standardı (1st moment + 2nd moment + bias correction)  
- Initialization: He (Kaiming) — ReLU'nun yarıya indirdiği varyansı 2× ile telafi eder  
- Regularization: Yok (saf aktivasyon etkisini görmek için)  
- Epoch: 20  

**Beklenti:** ReLU ve Leaky ReLU, Sigmoid ve Tanh'a göre daha hızlı yakınsayacak (vanishing gradient problemi nedeniyle).

In [ ]:
EPOCHS = 20
LR = 0.001

results_exp1 = {}
for act_name in ACTIVATION_FUNCTIONS:
    print(f"Training: {act_name}...", end=' ')
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    history = train_model(model, train_loader, val_loader, optimizer, criterion, EPOCHS)
    test_acc = evaluate_test(model, test_loader)
    history['test_acc'] = test_acc
    results_exp1[act_name] = history
    print(f"Val Acc: {max(history['val_acc']):.4f} | Test Acc: {test_acc:.4f}")

plot_results(results_exp1, 'Deney 1: Aktivasyon Fonksiyonu Karsilastirmasi')
summary_table(results_exp1)

**Analiz:**
- **Sigmoid/Tanh:** Doygunluk bölgelerinde gradyan ≈ 0 → vanishing gradient → yavaş öğrenme  
- **ReLU:** x > 0 için gradyan = 1 → hızlı yakınsama, ama **dead neuron** riski (x < 0 → gradyan = 0)  
- **Leaky ReLU:** x < 0 için küçük gradyan (0.01) → dead neuron sorununu hafifletir

---
## 5. DENEY 2: Aktivasyon × Optimizer Etkileşimi (Genişlik: Optimizasyon)

Derste gördüğümüz optimizasyon yöntemlerinin aktivasyon fonksiyonlarıyla nasıl etkileştiğini inceleyelim.  

| Optimizer | Özellik |
|-----------|--------|
| SGD | Temel güncelleme: w -= lr × ∇J |
| SGD + Momentum | Hız birikimi: v = βv + ∇J, sallantıyı azaltır |
| SGD + Nesterov | "Ghost jump" → geleceğe bakarak fren yapar, overshooting'i önler |
| RMSProp | Per-parametre lr: AdaGrad'ın düzeltilmiş hali (leaky bucket) |
| Adam | Momentum + RMSProp + bias correction |

In [ ]:
optimizers_config = {
    'SGD': lambda params: optim.SGD(params, lr=0.01),
    'SGD+Momentum': lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    'SGD+Nesterov': lambda params: optim.SGD(params, lr=0.01, momentum=0.9, nesterov=True),
    'RMSProp': lambda params: optim.RMSprop(params, lr=0.001),
    'Adam': lambda params: optim.Adam(params, lr=0.001),
}

test_activations = ['Sigmoid', 'ReLU']

results_exp2 = {}
for act_name in test_activations:
    for opt_name, opt_fn in optimizers_config.items():
        key = f"{act_name} + {opt_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method='he')
        optimizer = opt_fn(model.parameters())
        criterion = nn.CrossEntropyLoss()

        history = train_model(model, train_loader, val_loader, optimizer, criterion, EPOCHS)
        results_exp2[key] = history
        print(f"Val Acc: {max(history['val_acc']):.4f}")

plot_results(results_exp2, 'Deney 2: Aktivasyon x Optimizer')
summary_table(results_exp2)

**Analiz:**
- **Sigmoid + SGD:** En kötü kombinasyon — vanishing gradient + sabit learning rate  
- **Sigmoid + Adam:** Adaptive lr, Sigmoid'in yavaşlığını kısmen telafi eder  
- **ReLU + herhangi bir optimizer:** Güçlü gradyan akışı sayesinde tüm optimizer'lar iyi çalışır  
- **Nesterov vs Momentum:** Nesterov, "ghost jump" yaparak overshooting'i önler — Sigmoid'de daha belirgin fark  
- Bu sonuçlara göre Adam'ı diğer deneylerin varsayılan optimizer'ı olarak seçtik.

---
## 6. DENEY 3: Aktivasyon × Regularizasyon (Genişlik: Regularization)

Overfitting'i kontrol altına almak için farklı regularizasyon tekniklerinin aktivasyon fonksiyonlarıyla etkileşimi.

| Yöntem | Mekanizma | Bayesian Yorumu |
|--------|-----------|----------------|
| L2 (Weight Decay) | ½α‖w‖² → ağırlıkları küçültür | Gaussian prior |
| Dropout (p=0.3) | Rastgele nöron kapatma → 2ⁿ alt ağ ensemble'ı | — |
| Batch Normalization | Her katmanda normalize → kararlı dağılım (γ, β öğrenilebilir) | — |
| Label Smoothing (ε=0.1) | Hedef: 1-ε doğru, ε/(k-1) yanlış sınıflar → aşırı güveni engeller | — |

In [ ]:
criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
criterion_normal = nn.CrossEntropyLoss()

reg_configs = {
    'No Reg':        {'dropout': False, 'batchnorm': False, 'wd': 0,    'criterion': criterion_normal},
    'L2 (wd=1e-4)':  {'dropout': False, 'batchnorm': False, 'wd': 1e-4, 'criterion': criterion_normal},
    'Dropout(0.3)':  {'dropout': True,  'batchnorm': False, 'wd': 0,    'criterion': criterion_normal},
    'BatchNorm':     {'dropout': False, 'batchnorm': True,  'wd': 0,    'criterion': criterion_normal},
    'LabelSmooth':   {'dropout': False, 'batchnorm': False, 'wd': 0,    'criterion': criterion_smooth},
}

results_exp3 = {}
for act_name in ['Sigmoid', 'ReLU']:
    for reg_name, cfg in reg_configs.items():
        key = f"{act_name} + {reg_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name, use_dropout=cfg['dropout'], use_batchnorm=cfg['batchnorm'])
        model = init_weights(model, method='he')
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=cfg['wd'])

        history = train_model(model, train_loader, val_loader, optimizer, cfg['criterion'], EPOCHS)
        results_exp3[key] = history
        print(f"Val Acc: {max(history['val_acc']):.4f}")

plot_results(results_exp3, 'Deney 3: Aktivasyon x Regularizasyon')
summary_table(results_exp3)

**Analiz:**
- **Batch Normalization + Sigmoid:** BN, internal covariate shift'i azaltarak Sigmoid'in doygunluk sorununu önemli ölçüde hafifletir. Learnable γ ve β, ağın normalizasyonu "geri almasına" izin verir.  
- **Dropout:** Train loss'u yükseltir (öğrenmeyi zorlaştırır) ama generalization gap'i kapatır  
- **Label Smoothing:** ε=0.1 ReLU'da iyileştirme sağlar ama Sigmoid'de etkisiz — Sigmoid zaten belirsiz çıktılar üretir  
- **L2:** Ağırlıkları küçülterek Sigmoid'in doygunluk bölgesinden uzak kalmasına yardımcı olur

---
## 7. DENEY 4: Aktivasyon × Başlangıç Stratejisi (Genişlik: Initialization)

**Neden önemli?**
- Tüm ağırlıklar aynı → **simetri sorunu** (tüm nöronlar aynı şeyi öğrenir)  
- Ağırlıklar çok küçük → **vanishing gradients**  
- Ağırlıklar çok büyük → **exploding gradients**  

| Yöntem | Varyans | En Uygun Aktivasyon |
|--------|---------|---------------------|
| Xavier (Glorot) | 2/(n_in + n_out) | Sigmoid, Tanh |
| He (Kaiming) | 2/n_in | ReLU, Leaky ReLU |
| Random (σ=0.5) | Kontrolsüz | — (riskli) |
| Zeros | 0 | Hiçbiri (simetri kırılmaz) |

In [ ]:
init_methods = ['xavier', 'he', 'random', 'zeros']

results_exp4 = {}
for act_name in ACTIVATION_FUNCTIONS:
    for init_name in init_methods:
        key = f"{act_name} + {init_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method=init_name)
        optimizer = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()

        history = train_model(model, train_loader, val_loader, optimizer, criterion, 15)
        results_exp4[key] = history
        print(f"Val Acc: {max(history['val_acc']):.4f}")

plot_results(results_exp4, 'Deney 4: Aktivasyon x Initialization')
summary_table(results_exp4)

**Analiz:**
- **Xavier + Sigmoid/Tanh:** Doğru eşleşme — Xavier, sinyalin sabit kalmasını sağlar (linear-ish aktivasyonlar için)  
- **He + ReLU/LeakyReLU:** Doğru eşleşme — He, ReLU'nun yarıya indirdiği varyansı 2× ile telafi eder  
- **Zeros:** Tüm aktivasyonlarda tamamen başarısız → simetri kırılması ihlal edildi  
- **Random (σ=0.5):** Sigmoid'i çökertiyor (saturation), ReLU daha dayanıklı

---
## 8. DENEY 5: Gradyan Akış Analizi (Gradient Flow)

Backpropagation'da gradyanların katmanlar arası akışını doğrudan gözlemleyelim.  
8 katmanlı derin ağ ile vanishing gradient problemini belirgin hale getiriyoruz.  
**Vanishing gradient = derin katmanlarda gradyan normu ≈ 0 → öğrenme durur.**

In [ ]:
def get_gradient_norms(model, data_loader, criterion):
    model.train()
    X_batch, y_batch = next(iter(data_loader))
    X_batch = X_batch.view(-1, 784).to(device)
    y_batch = y_batch.to(device)
    model.zero_grad()
    loss = criterion(model(X_batch), y_batch)
    loss.backward()
    grad_norms, layer_names = [], []
    for name, param in model.named_parameters():
        if 'weight' in name and param.grad is not None:
            grad_norms.append(param.grad.norm().item())
            layer_names.append(name)
    return layer_names, grad_norms


deep_hidden = [512, 256, 128, 128, 64, 64, 32, 32]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for act_name in ACTIVATION_FUNCTIONS:
    torch.manual_seed(42)
    model = build_model(act_name, hidden_sizes=deep_hidden)
    model = init_weights(model, method='he')
    model.to(device)

    names, norms = get_gradient_norms(model, train_loader, nn.CrossEntropyLoss())
    axes[0].plot(range(len(norms)), norms, 'o-', label=act_name, color=COLORS[act_name], markersize=5)

    optimizer = optim.Adam(model.parameters(), lr=LR)
    history = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), 15)
    axes[1].plot(history['val_acc'], label=act_name, color=COLORS[act_name])

axes[0].set_xlabel('Katman (girise yakin -> cikisa yakin)')
axes[0].set_ylabel('Gradyan Normu')
axes[0].set_title('Gradyan Akisi: 8 Katmanli Ag (Baslangic)')
axes[0].legend(); axes[0].set_yscale('log')

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
axes[1].set_title('8 Katmanli Ag: Egitim Performansi')
axes[1].legend()

plt.tight_layout(); plt.show()
print("Sigmoid/Tanh: giris katmanlarina dogru gradyan ~300x duser (vanishing gradient)")
print("ReLU/LeakyReLU: gradyan akisi katmanlar boyunca stabil kalir")

---
## 9. DENEY 6: Depth vs Width (Genişlik: Week 3)

Her katman input space'i "katlar" → L katman sonrası 2^L bölge ayırt edilebilir.  
Ama bu avantaj yalnızca gradyanlar geriye akabiliyorsa işler. Sigmoid'in derinlikle ilişkisini test ediyoruz.

In [ ]:
architectures = {
    'Shallow (512,)': [512],
    'Medium (256,128)': [256, 128],
    'Deep (256,128,64,32)': [256, 128, 64, 32],
}

results_exp6 = {}
for act_name in ['Sigmoid', 'ReLU']:
    for arch_name, hidden in architectures.items():
        key = f"{act_name} + {arch_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name, hidden_sizes=hidden)
        model = init_weights(model, method='he')
        n_params = sum(p.numel() for p in model.parameters())
        optimizer = optim.Adam(model.parameters(), lr=LR)
        history = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), EPOCHS)
        test_acc = evaluate_test(model, test_loader)
        history['test_acc'] = test_acc
        results_exp6[key] = history
        print(f"Params: {n_params:,} | Test Acc: {test_acc:.4f}")

plot_results(results_exp6, 'Deney 6: Depth vs Width')
summary_table(results_exp6)

**Analiz:**
- **ReLU + derinlik:** Performans artar (daha fazla katman → daha fazla karar bölgesi)  
- **Sigmoid + derinlik:** Performans düşer — vanishing gradient erken katmanların öğrenmesini engeller  
- Bu, Week 3'teki "depth beats width" teorisinin yalnızca gradyanlar akabildiğinde geçerli olduğunu gösterir

---
## 10. DENEY 7: Early Stopping (Genişlik: Week 4)

Early stopping, validation loss iyileşmediğinde eğitimi durdurarak overfitting'i önler.  
Aktivasyon fonksiyonunun overfitting hızını doğrudan etkiler:  
- **ReLU:** Güçlü gradyanlar → hızlı öğrenme → hızlı overfitting (düşük bias, yüksek variance)  
- **Sigmoid:** Zayıf gradyanlar → yavaş öğrenme → underfitting (yüksek bias, düşük variance)

In [ ]:
results_exp7 = {}
for act_name in ACTIVATION_FUNCTIONS:
    print(f"Training: {act_name} (max 60 epochs, patience=7)...", end=' ')
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)

    history = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(),
                          epochs=60, early_stop_patience=7)
    stopped_at = len(history['train_loss'])
    test_acc = evaluate_test(model, test_loader)
    history['test_acc'] = test_acc
    results_exp7[act_name] = history
    print(f"Stopped at epoch {stopped_at} | Test Acc: {test_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, hist in results_exp7.items():
    epochs_ran = len(hist['train_loss'])
    ax.plot(hist['train_loss'], '--', color=COLORS[name], alpha=0.4)
    ax.plot(hist['val_loss'], color=COLORS[name], label=f"{name} (stop@{epochs_ran})")
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Early Stopping: Train (kesikli) vs Val (duz) Loss')
ax.legend(); plt.tight_layout(); plt.show()

---
## 11. DENEY 8: Learning Rate Scheduling (Genişlik: Week 5)

Adaptive LR schedule, eğitimin başında büyük adımlar (exploration) sonunda küçük adımlar (refinement) atılmasını sağlar.  
Sigmoid'in zaten çok küçük gradyanları olduğu için LR azaltmanın etkisi sınırlı olabilir.

In [ ]:
results_exp8 = {}
for act_name in ['Sigmoid', 'ReLU']:
    for sched_name in ['Constant', 'StepLR', 'CosineAnnealing']:
        key = f"{act_name} + {sched_name}"
        print(f"Training: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method='he')
        optimizer = optim.Adam(model.parameters(), lr=LR)

        scheduler = None
        if sched_name == 'StepLR':
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
        elif sched_name == 'CosineAnnealing':
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        history = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(),
                              EPOCHS, scheduler=scheduler)
        results_exp8[key] = history
        print(f"Val Acc: {max(history['val_acc']):.4f}")

plot_results(results_exp8, 'Deney 8: LR Scheduling')
summary_table(results_exp8)

**Analiz:**
- **Cosine annealing + ReLU:** Marjinal iyileşme sağlar — refinement fazı faydalı  
- **Cosine annealing + Sigmoid:** Etkisiz — gradyanlar zaten doygunlukta sıfıra yakın, LR küçültmek yardımcı olmaz  
- Adam'ın kendi adaptive rate'leri zaten global scheduling'in çoğunu karşılıyor

---
## 12. Ölü Nöron Analizi (Dead Neuron Problem)

ReLU'da x < 0 → çıkış = 0 ve gradyan = 0. Eğer bir nöronun girdisi sürekli negatif kalırsa, o nöron **ölür** ve bir daha öğrenemez. Leaky ReLU, küçük bir negatif eğim ile bunu önler.

In [ ]:
def count_dead_neurons(model, data_loader, num_batches=10):
    model.eval()
    activations, counts = {}, {}
    hooks = []
    layer_idx = 0
    for module in model.modules():
        if isinstance(module, (nn.ReLU, nn.LeakyReLU)):
            idx = layer_idx
            def hook_fn(m, inp, out, idx=idx):
                act = (out > 0).float().mean(dim=0)
                if idx not in activations:
                    activations[idx] = act; counts[idx] = 1
                else:
                    activations[idx] += act; counts[idx] += 1
            hooks.append(module.register_forward_hook(hook_fn))
            layer_idx += 1
    with torch.no_grad():
        for i, (X_batch, _) in enumerate(data_loader):
            if i >= num_batches: break
            model(X_batch.view(-1, 784).to(device))
    for h in hooks: h.remove()
    results = {}
    for idx in activations:
        avg_act = activations[idx] / counts[idx]
        dead = (avg_act < 0.01).sum().item()
        total = avg_act.numel()
        results[f'Layer {idx}'] = (dead, total, dead/total*100)
    return results

for act_name in ['ReLU', 'LeakyReLU']:
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)
    train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), epochs=10)
    dead_info = count_dead_neurons(model, train_loader)
    print(f"\n{act_name} — Dead Neuron Analysis (10 epoch sonrasi):")
    for layer, (dead, total, pct) in dead_info.items():
        print(f"  {layer}: {dead}/{total} dead ({pct:.1f}%)")

---
## 13. Bias-Variance Perspektifi

Train loss ile val loss arasındaki **generalization gap**, modelin variance'ını yansıtır.  
- **Yüksek bias** (underfitting): Her iki loss yüksek → model kapasitesi yetersiz  
- **Yüksek variance** (overfitting): Train loss düşük, val loss yüksek → regularization gerekli

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, act_name in enumerate(ACTIVATION_FUNCTIONS):
    hist = results_exp1[act_name]
    axes[i].plot(hist['train_loss'], label='Train Loss', linewidth=2)
    axes[i].plot(hist['val_loss'], label='Val Loss', linewidth=2)
    axes[i].fill_between(range(EPOCHS), hist['train_loss'], hist['val_loss'], alpha=0.2, color='red')
    gap = hist['val_loss'][-1] - hist['train_loss'][-1]
    axes[i].set_title(f'{act_name}\nGen. Gap = {gap:.3f}')
    axes[i].set_xlabel('Epoch'); axes[i].set_ylabel('Loss'); axes[i].legend(fontsize=8)
plt.suptitle('Bias-Variance: Generalization Gap (kirmizi alan = overfitting gostergesi)', fontsize=13)
plt.tight_layout(); plt.show()

---
## 14. L1 vs L2 Regularizasyon Detaylı Karşılaştırma

- **L2 (Ridge):** J(w) = Loss + α/2 · ‖w‖² → ağırlıkları küçültür ama sıfırlamaz → Gaussian prior  
- **L1 (Lasso):** J(w) = Loss + α · ‖w‖₁ → ağırlıkları **tam sıfıra** iter → Laplace prior → feature selection

In [ ]:
def train_with_l1(model, train_loader, val_loader, optimizer, criterion, l1_lambda, epochs=15):
    model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'grad_norms': []}
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.view(-1, 784).to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                val_loss += criterion(logits, y_batch).item()
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)
        history['train_loss'].append(running_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(correct / total)
        history['grad_norms'].append(0)
    return history

results_l1l2 = {}
for reg_type, config in [('No Reg', {'wd': 0, 'l1': 0}),
                          ('L2 (1e-4)', {'wd': 1e-4, 'l1': 0}),
                          ('L1 (1e-5)', {'wd': 0, 'l1': 1e-5})]:
    torch.manual_seed(42)
    model = build_model('ReLU')
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=config['wd'])
    if config['l1'] > 0:
        history = train_with_l1(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), config['l1'], EPOCHS)
    else:
        history = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), EPOCHS)
    results_l1l2[reg_type] = history

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (reg_type, config) in enumerate([('No Reg', {'wd': 0, 'l1': 0}),
                                         ('L2', {'wd': 1e-4, 'l1': 0}),
                                         ('L1', {'wd': 0, 'l1': 1e-5})]):
    torch.manual_seed(42)
    model = build_model('ReLU')
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=config['wd'])
    if config['l1'] > 0:
        train_with_l1(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), config['l1'], EPOCHS)
    else:
        train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), EPOCHS)
    all_weights = torch.cat([p.data.cpu().flatten() for p in model.parameters()])
    axes[i].hist(all_weights.numpy(), bins=100, density=True, alpha=0.7)
    axes[i].set_title(f'{reg_type}\nZero ratio: {(all_weights.abs() < 0.001).float().mean():.1%}')
    axes[i].set_xlabel('Weight value'); axes[i].set_xlim(-0.5, 0.5)
plt.suptitle('L1 vs L2: Weight Distributions (L1 -> sparsity, L2 -> shrinkage)', fontsize=13)
plt.tight_layout(); plt.show()
summary_table(results_l1l2)

---
## 15. Sonuç Özet Tablosu

In [ ]:
print("=" * 90)
print("GENEL SONUC OZETI")
print("=" * 90)
print("\n[Deney 1] Aktivasyon Fonksiyonu Karsilastirmasi (Derinlik):")
summary_table(results_exp1)
print("\n[Deney 2] Aktivasyon x Optimizer:")
summary_table(results_exp2)
print("\n[Deney 3] Aktivasyon x Regularizasyon:")
summary_table(results_exp3)
print("\n[Deney 4] Aktivasyon x Initialization:")
summary_table(results_exp4)
print("\n[Deney 6] Depth vs Width:")
summary_table(results_exp6)
print("\n[Deney 7] Early Stopping:")
for name, hist in results_exp7.items():
    print(f"  {name}: stopped at epoch {len(hist['train_loss'])}, test acc = {hist.get('test_acc', 'N/A')}")
print("\n[Deney 8] LR Scheduling:")
summary_table(results_exp8)

## Sonuçlar

### Derinlik: Aktivasyon Fonksiyonları
1. **ReLU/Leaky ReLU** hemen her konfigürasyonda Sigmoid/Tanh'a göre daha hızlı yakınsama ve daha yüksek doğruluk sağlar.
2. **Sigmoid** en yavaş öğrenme ve en düşük performansı gösterir (vanishing gradient).
3. **Leaky ReLU**, ReLU'nun dead neuron sorununu hafifletir.

### Genişlik: Diğer Metodolojiler
4. **Optimizer etkisi:** Adam, tüm aktivasyonlarda en stabil performansı verir. SGD, momentum olmadan Sigmoid ile çok zorlanır.
5. **Batch Normalization**, Sigmoid ile birlikte kullanıldığında en büyük tek müdahale iyileştirmesini sağlar.
6. **Initialization:** He init ReLU ile, Xavier init Sigmoid/Tanh ile doğru eşleşme sağlar. Zeros tamamen başarısız olur.
7. **Depth:** ReLU derinlikle iyileşir, Sigmoid derinlikle kötüleşir — vanishing gradient bottleneck.
8. **Early Stopping:** ReLU daha erken durdurulur (hızlı overfitting), Sigmoid uzun süre çalışır (underfitting).
9. **LR Scheduling:** Cosine annealing ReLU'ya marjinal katkı sağlar, Sigmoid'e etkisiz.
10. **L1 → seyreklik** (feature selection); **L2 → genel küçültme** (tüm öznitelikler önemli olduğunda).

### Pratik Öneri
- Varsayılan: **ReLU + He init + Adam + BatchNorm** → güvenli ve güçlü başlangıç noktası.
- Ölü nöron sorunu varsa: **Leaky ReLU** deneyin.
- Overfitting varsa: **Dropout + L2** kombinasyonu etkili.

---

## Kapsanan Ders Konuları (Checklist)

| Hafta | Konu | Nerede Kullanıldı |
|-------|------|-------------------|
| 2 | ML Temelleri: i.i.d., Generalization, Bias-Variance | Veri seti, Deney 1, Bölüm 13 |
| 2 | MLE ↔ Loss Function bağlantısı | Cross-entropy = Categorical MLE (Bölüm 3) |
| 2 | SGD ve Minibatch | Tüm eğitim döngüleri (batch_size=128) |
| 2 | Regularization temelleri (L1, L2) | Deney 3, Bölüm 14 (Bayesian yorum dahil) |
| 3 | Feedforward Networks, Depth vs Width | Model mimarisi, **Deney 6** |
| 3 | Softmax, Cross-Entropy (BCE/CCE) | Loss function seçimi (CCE, 10 sınıf) |
| 3 | Backpropagation | Eğitim döngüsü, Gradyan akış analizi (Deney 5) |
| 3 | Aktivasyon Fonksiyonları | **ANA KONU** — Tüm deneyler |
| 4 | L2 Weight Decay | Deney 3 |
| 4 | L1 Lasso (seyreklik) | Bölüm 14 |
| 4 | Dropout | Deney 3 |
| 4 | Label Smoothing | Deney 3 |
| 4 | Batch Normalization | Deney 3 |
| 4 | Early Stopping | **Deney 7** |
| 5 | Initialization (Xavier, He) | Deney 4 |
| 5 | Optimizers (SGD, Momentum, Nesterov, RMSProp, Adam) | Deney 2 |
| 5 | Vanishing/Exploding Gradients | Deney 5 (8 katmanlı ağ) |
| 5 | LR Scheduling (StepLR, Cosine) | **Deney 8** |